**Programmer:** python_scripts (Abhijith Warrier)

**PYTHON SCRIPT TO UNDERSTAND METACLASSES — PYTHON'S CLASS FACTORY MECHANISM & THE POWER BEHIND ADVANCED FRAMEWORKS 🧠🐍**

In Python:
- *Everything is an object*
- *Every object has a type*
- And *classes themselves are objects*, created by **metaclasses**

This DeepCut explores:
- How `type()` creates classes dynamically
- The difference between `__new__` and `__init__`
- How metaclasses are used for *automatic registration*, *validation*, and *framework internals* (like Django, SQLAlchemy, Pydantic)

---

## 📦 Import Standard Library

In [1]:
import inspect

---

## 🧩 Snippet 1 — type() is the real class creator

In Python, classes are created by calling `type(name, bases, attrs)`.

This means we can manually construct a class without `class` keyword.

In [2]:
# Create a class manually using type()
Person = type(
    "Person",               # class name
    (object,),              # base classes
    {"x": 42}               # attributes
)

p = Person()
print(Person, p.x)

<class '__main__.Person'> 42


---

## 🧠 Snippet 2 — __new__ creates the class, __init__ configures it

For metaclasses:

- `__new__` → called *before* class creation; returns the new class object
- `__init__` → called *after* class is created; initializes metadata

`__new__` is where metaclass magic happens.

In [3]:
class Meta(type):
    def __new__(mcls, name, bases, attrs):
        print(f"[__new__] Creating class {name}")
        cls = super().__new__(mcls, name, bases, attrs)
        return cls

    def __init__(cls, name, bases, attrs):
        print(f"[__init__] Initializing class {name}")
        super().__init__(name, bases, attrs)

class Example(metaclass=Meta):
    pass

[__new__] Creating class Example
[__init__] Initializing class Example


---

## 🏗️ Snippet 3 — Framework-style auto-registration using metaclasses

Many frameworks (Django, SQLAlchemy, Pydantic) use metaclasses to:
- register subclasses
- auto-generate fields
- enforce rules
- perform analysis during class creation

In [4]:
REGISTRY = {}

class AutoRegister(type):
    def __new__(mcls, name, bases, attrs):
        cls = super().__new__(mcls, name, bases, attrs)
        if name != "Base":
            REGISTRY[name] = cls
        return cls

class Base(metaclass=AutoRegister):
    pass

class User(Base):
    pass

class Product(Base):
    pass

print(REGISTRY)

{'User': <class '__main__.User'>, 'Product': <class '__main__.Product'>}


---

## 🔧 Snippet 4 — Metaclasses can inject attributes or modify classes

This is how frameworks add:
- validators
- annotations
- special methods
- auto-generated fields

In [5]:
class AutoStr(type):
    def __new__(mcls, name, bases, attrs):
        if "__str__" not in attrs:
            def __str__(self):
                return f"<{name} instance>"
            attrs["__str__"] = __str__
        return super().__new__(mcls, name, bases, attrs)

class Node(metaclass=AutoStr):
    pass

n = Node()
print(n)

<Node instance>


---

## 🧬 Snippet 5 — Metaclasses can validate class structure

This pattern is used for:
- enforcing required attributes
- checking method signatures
- validating schema definitions

In [6]:
class RequiresSave(type):
    def __new__(mcls, name, bases, attrs):
        if name != "BaseModel" and "save" not in attrs:
            raise TypeError(f"{name} must define save()")
        return super().__new__(mcls, name, bases, attrs)

class BaseModel(metaclass=RequiresSave):
    pass

class Model(BaseModel):
    def save(self):
        print("Saving...")

# Uncommenting this will error:
# class BadModel(BaseModel):
#     pass

---

## ✅ One-liner Takeaway

**Metaclasses are class factories — they let you intercept and customize how classes are created, enabling registration, validation, and framework-level magic.**

---